# Team 7 · Export the Material Finder v2 (for the app and Hugging Face)

This notebook packages the winner of `improvements_round2.ipynb` and `top_materials.ipynb` for the model server:

**photo → centre crop (middle half) → 224 px → frozen DINOv2-small (768 features) → our feature autoencoder → 32-number code (length 1) → cosine search in a 3,900-patch gallery → ranked materials**

One ONNX file does everything after the resize: ImageNet normalisation, DINOv2, standardising the features, the encoder (→ `code`) and the decoder (→ `unusual_score`, the feature reconstruction error). The server only needs Pillow, NumPy and ONNX Runtime.

All final numbers below are measured **with the ONNX file on the real images**, exactly as the server will run it.

**Needs:** `data/minc-2500/data/*.parquet`, the caches `data/minc_round2_*.npz` and `data/minc_extra_train.npz` (made by the improvement notebooks). DINOv2-small is downloaded from Hugging Face on first use.

In [1]:
import io
import json
import shutil
import time
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd
import polars as pl
import torch
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModel

AE_DIR = Path("..")
DATA_DIR = AE_DIR / ".." / "data"
MINC_DIR = DATA_DIR / "minc-2500" / "data"
FIG_DIR = AE_DIR / "figures"
MODEL_DIR = AE_DIR / "models"          # goes to Hugging Face
APP_DIR = AE_DIR / "model_app"         # what the Tuesday server needs
for folder in [MODEL_DIR / "gallery_thumbs", MODEL_DIR / "figures", MODEL_DIR / "examples", APP_DIR / "examples"]:
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ALL_CLASSES = ["brick", "carpet", "ceramic", "fabric", "foliage", "food", "glass", "hair", "leather",
               "metal", "mirror", "other", "painted", "paper", "plastic", "polishedstone", "skin", "sky",
               "stone", "tile", "wallpaper", "water", "wood"]
BUILDING = ["brick", "carpet", "ceramic", "glass", "metal", "mirror", "painted", "plastic",
            "polishedstone", "stone", "tile", "wallpaper", "wood"]
BUILDING_IDS = [ALL_CLASSES.index(c) for c in BUILDING]
OTHER_IDS = [i for i, c in enumerate(ALL_CLASSES) if c not in BUILDING]
remap = {ALL_CLASSES.index(c): k for k, c in enumerate(BUILDING)}
N_CLASSES = len(BUILDING)
print(f"PyTorch {torch.__version__}, ONNX Runtime {ort.__version__}, device {DEVICE}")

/Users/snehashenoy/Downloads/group7_day1_assigment/hackathon-team7-MINC_2500_material_patches/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch 2.14.1, ONNX Runtime 1.31.1, device mps


# 1. Train the feature autoencoder (same recipe and seed as before)

Trained on the cached DINOv2 centre-crop features of the 27,625 training patches: 768 → 256 → **32** → 256 → 768, denoising (noise 0.5) + weight decay 1e-4, 30 epochs. No material labels.

In [2]:
cache = {s: np.load(DATA_DIR / f"minc_round2_{s}.npz") for s in ["train", "val", "test", "test_other"]}
F_train = cache["train"]["dino_centre"]
y_train = np.array([remap[v] for v in cache["train"]["y"]])
y_test = np.array([remap[v] for v in cache["test"]["y"]])
FEAT_MU, FEAT_SD = F_train.mean(0), F_train.std(0) + 1e-6


class FeatureAE(nn.Module):
    def __init__(self, n_in=768, latent_dim=32, hidden=256):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden), nn.ReLU(), nn.Linear(hidden, n_in))

    def forward(self, x):
        return self.decoder(self.encoder(x))


Fs_train = ((F_train - FEAT_MU) / FEAT_SD).astype(np.float32)
torch.manual_seed(RANDOM_STATE)
ae = FeatureAE().to(DEVICE)
opt = torch.optim.AdamW(ae.parameters(), lr=1e-3, weight_decay=1e-4)
loader = DataLoader(TensorDataset(torch.from_numpy(Fs_train)), batch_size=256, shuffle=True,
                    generator=torch.Generator().manual_seed(RANDOM_STATE))
start = time.time()
for epoch in range(30):
    ae.train()
    for (x,) in loader:
        x = x.to(DEVICE)
        loss = nn.functional.mse_loss(ae(x + 0.5 * torch.randn_like(x)), x)
        opt.zero_grad(); loss.backward(); opt.step()
ae = ae.cpu().eval()
print(f"trained in {time.time() - start:.0f} s, {sum(p.numel() for p in ae.parameters()):,} parameters")

# The app's gallery: 300 random training patches per material (same seed and rule as top_materials.ipynb)
rng = np.random.default_rng(RANDOM_STATE)
gallery_idx = np.concatenate([rng.choice(np.flatnonzero(y_train == k), 300, replace=False) for k in range(N_CLASSES)])
y_gal = y_train[gallery_idx]
print("gallery:", len(gallery_idx))

trained in 9 s, 410,912 parameters
gallery: 3900


# 2. One model for the server: `MaterialFinder`

Input: `image`, float32 (batch, 3, 224, 224), pixels 0–1, already cropped and resized. Outputs:
- `code` (batch, 32): the fingerprint, scaled to length 1, so Euclidean search in the gallery = cosine search;
- `unusual_score` (batch,): how badly the autoencoder rebuilds the DINOv2 features (high = unlike building materials).

In [3]:
class MaterialFinder(nn.Module):
    def __init__(self, dino, ae, feat_mu, feat_sd):
        super().__init__()
        self.dino, self.ae = dino, ae
        self.register_buffer("img_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("img_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.register_buffer("feat_mu", torch.from_numpy(feat_mu).float())
        self.register_buffer("feat_sd", torch.from_numpy(feat_sd).float())

    def forward(self, image):
        x = (image - self.img_mean) / self.img_std
        h = self.dino(pixel_values=x).last_hidden_state                     # (B, 1 + 256, 384)
        f = torch.cat([h[:, 0], h[:, 1:].mean(1)], dim=1)                   # class token + mean patch token
        fs = (f - self.feat_mu) / self.feat_sd
        z = self.ae.encoder(fs)
        unusual_score = ((self.ae.decoder(z) - fs) ** 2).mean(1)
        code = z / z.norm(dim=1, keepdim=True)
        return code, unusual_score


dino = AutoModel.from_pretrained("facebook/dinov2-small", attn_implementation="eager").eval()
finder = MaterialFinder(dino, ae, FEAT_MU, FEAT_SD).eval()

ONNX_PATH = MODEL_DIR / "material_finder.onnx"
start = time.time()
torch.onnx.export(finder, (torch.rand(2, 3, 224, 224),), str(ONNX_PATH), dynamo=False, opset_version=17,
                  input_names=["image"], output_names=["code", "unusual_score"],
                  dynamic_axes={"image": {0: "batch"}, "code": {0: "batch"}, "unusual_score": {0: "batch"}})
print(f"exported in {time.time() - start:.0f} s: {ONNX_PATH.name}, {ONNX_PATH.stat().st_size / 1e6:.1f} MB")
session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])

W1117 10:44:51.384000 1493 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 7439.02it/s]


/var/folders/7x/4mr6dq0s3ll7hrz5x39p97680000gp/T/ipykernel_1493/319757337.py:26: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(finder, (torch.rand(2, 3, 224, 224),), str(ONNX_PATH), dynamo=False, opset_version=17,
/Users/snehashenoy/Downloads/group7_day1_assigment/hackathon-team7-MINC_2500_material_patches/.venv/lib/python3.12/site-packages/transformers/models/dinov2/modeling_dinov2.py:62: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to o

exported in 1 s: material_finder.onnx, 90.1 MB


# 3. Preprocessing (the server must do exactly this) and the ONNX check

In [4]:
def preprocess(image: Image.Image) -> np.ndarray:
    # RGB -> centre square -> its middle half -> 224 x 224 (bilinear) -> 0-1 -> (1, 3, 224, 224) float32
    image = image.convert("RGB")
    w, h = image.size
    side = min(w, h)
    left, top = (w - side) // 2, (h - side) // 2
    image = image.crop((left, top, left + side, top + side))
    half = int(side * 0.5)
    off = (side - half) // 2
    image = image.crop((off, off, off + half, off + half)).resize((224, 224), Image.BILINEAR)
    return (np.asarray(image, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None]


def load_bytes(split, label_ids):
    files = sorted(MINC_DIR.glob(f"{split}-*.parquet"))
    df = (pl.scan_parquet(files).select(pl.col("image").struct.field("bytes").alias("bytes"), pl.col("label"))
          .filter(pl.col("label").is_in(label_ids)).collect())
    return df["bytes"].to_list(), df["label"].to_numpy()


def run_onnx(byte_list, batch_size=32):
    codes, scores = [], []
    for i in range(0, len(byte_list), batch_size):
        x = np.concatenate([preprocess(Image.open(io.BytesIO(b))) for b in byte_list[i:i + batch_size]])
        c, s = session.run(None, {"image": x})
        codes.append(c); scores.append(s)
    return np.concatenate(codes), np.concatenate(scores)


raw = {}
for split, ids, name in [("train", BUILDING_IDS, "train"), ("validation", BUILDING_IDS, "val"),
                         ("test", BUILDING_IDS, "test"), ("test", OTHER_IDS, "test_other")]:
    raw[name], labels = load_bytes(split, ids)
    assert (labels == cache[name]["y"]).all(), f"order of {name} differs from the cache"
print({k: len(v) for k, v in raw.items()})

# ONNX vs PyTorch on the same 16 test photos
x = np.concatenate([preprocess(Image.open(io.BytesIO(b))) for b in raw["test"][:16]])
with torch.no_grad():
    c_pt, s_pt = finder(torch.from_numpy(x))
c_ox, s_ox = session.run(None, {"image": x})
print(f"code: max difference {np.abs(c_ox - c_pt.numpy()).max():.2e} | "
      f"unusual score: max relative difference {np.abs(s_ox / s_pt.numpy() - 1).max():.2e}")
assert np.allclose(c_ox, c_pt.numpy(), atol=1e-3)

{'train': 27625, 'val': 1625, 'test': 3250, 'test_other': 2500}


code: max difference 1.28e-06 | unusual score: max relative difference 2.92e-06


# 4. Gallery, threshold and final numbers (all with the ONNX file)

About 11,000 photos through DINOv2 on the CPU: a few minutes.

In [5]:
start = time.time()
Z_gal, _ = run_onnx([raw["train"][i] for i in gallery_idx])
_, s_val = run_onnx(raw["val"])
Z_test, s_test = run_onnx(raw["test"])
_, s_other = run_onnx(raw["test_other"])
print(f"ONNX on {len(gallery_idx) + len(raw['val']) + len(raw['test']) + len(raw['test_other']):,} photos "
      f"in {time.time() - start:.0f} s")

THRESHOLD = float(np.percentile(s_val, 95))
K, MIN_SHARE = 5, 0.2
dist, nbr = NearestNeighbors(n_neighbors=K).fit(Z_gal).kneighbors(Z_test)
votes = y_gal[nbr]
shares = np.stack([(votes == k).mean(1) for k in range(N_CLASSES)], axis=1)
closest = np.where(votes[:, :, None] == np.arange(N_CLASSES), dist[:, :, None], np.inf).min(1)
order = np.argsort(-(shares - 1e-6 * np.where(np.isinf(closest), 0, closest)), axis=1, kind="stable")
top = lambda k: ((order[:, :k] == y_test[:, None]) & (np.take_along_axis(shares, order[:, :k], 1) > 0)).any(1).mean()
shown = np.take_along_axis(shares, order[:, :3], 1) >= MIN_SHARE
shown[:, 0] = True

results = {
    "precision_at_5_gallery": float((votes == y_test[:, None]).mean()),
    "precision_at_5_full_train": 0.647,          # improvements_round2.ipynb (search in all 27,625 training patches)
    "top1_accuracy": float(top(1)), "top2_accuracy": float(top(2)), "top3_accuracy": float(top(3)),
    "right_material_shown_min_share_0_2": float(((order[:, :3] == y_test[:, None]) & shown).any(1).mean()),
    "suggestions_shown_avg": float(shown.sum(1).mean()),
    "unusual_roc_auc": float(roc_auc_score(np.r_[np.zeros(len(s_test)), np.ones(len(s_other))], np.r_[s_test, s_other])),
    "unusual_flagged_building": float((s_test > THRESHOLD).mean()),
    "unusual_flagged_other": float((s_other > THRESHOLD).mean()),
}
pd.Series(results).to_frame("value")

ONNX on 11,275 photos in 418 s


,value
precision_at_5_gallery,0.607015
precision_at_5_full_train,0.647000
top1_accuracy,0.698462
top2_accuracy,0.812000
top3_accuracy,0.846462
right_material_shown_min_share_0_2,0.846462
suggestions_shown_avg,2.005538
unusual_roc_auc,0.817759
unusual_flagged_building,0.050769
unusual_flagged_other,0.412800


In [6]:
# Unusual score per other class: which non-building photos get flagged?
other_names = [ALL_CLASSES[v] for v in cache["test_other"]["y"]]
(pd.DataFrame({"class": other_names, "flagged": s_other > THRESHOLD})
 .groupby("class")["flagged"].mean().sort_values().to_frame("share flagged").T)

class,paper,fabric,leather,food,other,water,foliage,sky,hair,skin
share flagged,0.116,0.204,0.232,0.264,0.316,0.444,0.492,0.624,0.684,0.752


# 5. Write the files for Hugging Face (`models/`) and the server (`model_app/`)

In [7]:
# Gallery thumbnails: 128 px full patches (what the user sees in the app)
full128 = np.load(DATA_DIR / "minc_extra_train.npz")["full128"]
meta = []
for pos, i in enumerate(gallery_idx):
    path = f"gallery_thumbs/{pos:05d}.jpg"
    Image.fromarray(full128[i]).save(MODEL_DIR / path, quality=88)
    meta.append({"id": int(i), "material": BUILDING[y_train[i]], "thumb": path})
np.save(MODEL_DIR / "gallery_latents.npy", Z_gal.astype(np.float32))
(MODEL_DIR / "gallery_meta.json").write_text(json.dumps(meta))

torch.save({"feature_ae": ae.state_dict(), "feat_mu": FEAT_MU, "feat_sd": FEAT_SD,
            "backbone": "facebook/dinov2-small"}, MODEL_DIR / "material_finder_ae.pt")

config = {
    "model": "material_finder.onnx",
    "version": 2,
    "setup": "frozen DINOv2-small (centre crop, 224 px) + feature autoencoder 768-256-32, cosine distance",
    "backbone": "facebook/dinov2-small (frozen, not trained by us)",
    "latent_dim": 32,
    "image_size": 224,
    "preprocessing": "RGB, centre square, keep its middle half, resize to 224x224 (PIL bilinear), pixels / 255 -> 0-1, "
                     "shape (1, 3, 224, 224) float32. ImageNet normalisation happens inside the ONNX file.",
    "inputs": {"image": "float32 (batch, 3, 224, 224), values 0-1"},
    "outputs": {"code": "float32 (batch, 32), length 1", "unusual_score": "float32 (batch,)"},
    "classes": BUILDING,
    "similarity": "Euclidean distance between length-1 codes (= cosine); 5 nearest gallery patches",
    "neighbours": K,
    "answer": "ranked materials by share of the 5 neighbours; show materials with share >= min_share, at most 3",
    "min_share": MIN_SHARE,
    "gallery_size": len(gallery_idx),
    "unusual_threshold": THRESHOLD,
    "unusual_threshold_rule": "95th percentile of unusual_score on validation building patches",
    "trained_parameters": sum(p.numel() for p in ae.parameters()),
    "results": results,
    "dataset": "MINC-2500 (mcimpoi/minc-2500_split_1), CC BY 4.0, Bell et al. 2015 CVPR",
}
(MODEL_DIR / "config.json").write_text(json.dumps(config, indent=2))

for name in ["01_samples", "09_improvements_precision", "12_round2_precision", "13_round2_per_material",
             "15_top_materials_accuracy", "16_top_materials_calibration", "17_top_materials_examples",
             "18_centre_crop_similarity"]:
    shutil.copy(FIG_DIR / f"{name}.png", MODEL_DIR / "figures" / f"{name}.png")

# Example photos: one test patch per material for 4 materials (original 362 px JPEGs)
for m in ["brick", "carpet", "tile", "wood"]:
    i = int(np.flatnonzero(y_test == BUILDING.index(m))[0])
    (MODEL_DIR / "examples" / f"{m}.jpg").write_bytes(raw["test"][i])

for f in ["material_finder.onnx", "config.json"]:
    shutil.copy(MODEL_DIR / f, APP_DIR / f)
for f in (MODEL_DIR / "examples").glob("*.jpg"):
    shutil.copy(f, APP_DIR / "examples" / f.name)
for p in sorted(MODEL_DIR.rglob("*")):
    if p.is_file() and "gallery_thumbs" not in p.parts:
        print(f"{p.relative_to(MODEL_DIR)}: {p.stat().st_size / 1e6:.2f} MB")
print(f"gallery_thumbs/: {len(list((MODEL_DIR / 'gallery_thumbs').glob('*.jpg')))} files")

config.json: 0.00 MB
examples/brick.jpg: 0.03 MB
examples/carpet.jpg: 0.03 MB
examples/tile.jpg: 0.03 MB
examples/wood.jpg: 0.04 MB
figures/01_samples.png: 0.55 MB
figures/09_improvements_precision.png: 0.13 MB
figures/12_round2_precision.png: 0.08 MB
figures/13_round2_per_material.png: 0.05 MB
figures/15_top_materials_accuracy.png: 0.06 MB
figures/16_top_materials_calibration.png: 0.05 MB
figures/17_top_materials_examples.png: 1.29 MB
figures/18_centre_crop_similarity.png: 1.11 MB
gallery_latents.npy: 0.50 MB
gallery_meta.json: 0.29 MB
material_finder.onnx: 90.05 MB
material_finder_ae.pt: 1.66 MB
gallery_thumbs/: 3900 files


# 6. Final check: the model card's "How to use" code on the example photos

In [8]:
gallery = np.load(MODEL_DIR / "gallery_latents.npy")
meta = json.loads((MODEL_DIR / "gallery_meta.json").read_text())
cfg = json.loads((MODEL_DIR / "config.json").read_text())

for photo in sorted((MODEL_DIR / "examples").glob("*.jpg")):
    code_, score = session.run(None, {"image": preprocess(Image.open(photo))})
    distances = np.linalg.norm(gallery - code_, axis=1)
    top5 = np.argsort(distances)[:5]
    materials = [meta[i]["material"] for i in top5]
    ranked = sorted(set(materials), key=lambda m: (-materials.count(m), materials.index(m)))
    answer = " · ".join(f"{m} {materials.count(m) / 5:.0%}" for m in ranked[:3] if materials.count(m) / 5 >= cfg["min_share"])
    print(f"{photo.name:11s} -> {answer:40s} unusual: {bool(score[0] > cfg['unusual_threshold'])}")

brick.jpg   -> brick 100%                               unusual: False
carpet.jpg  -> carpet 40% · wood 20% · tile 20%         unusual: False


tile.jpg    -> tile 80% · stone 20%                     unusual: False
wood.jpg    -> painted 40% · ceramic 40% · metal 20%    unusual: False
